# Convolutional Neural Network Review

A CNN is built by blocks.

## 1. Convolution (Conv)

Takes a small filter (kernel) a slides the whole images, in each position performs

`multiply + sum`

this produces a number, and by moving generates a full map (feature map)

<div>
    <img src='../images/CONV2D.png' width="600">
</div>

With an input image of size $(h, w)$ and a kernel of size $(k_h, k_w)$, the output dimensions are given by:

$$
o_h = h - k_h + 1 \\
o_w = w - k_w + 1
$$

We will derive this formula in the next section. For now, it can be understood intuitively from how the kernel slides over the input.

In [ ]:
import torch

x = torch.Tensor([
    [45, 12, 5, 17],
    [22, 10, 35, 6],
    [88, 26, 51, 19],
    [9, 77, 42, 3]
])

f = torch.Tensor([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

def conv(x, f):
    # Input shape
    h, w = x.shape
    # Kernel shape
    k_h, k_w = f.shape
    # Output
    o_h = h - k_h + 1
    o_w = w - k_w + 1
    output = torch.zeros(o_h, o_w)
    for i in range(o_h):
        for j in range(o_w):
            # Extraer parche
            patch = x[i:i+k_h, j:j+k_w]
            # Multiplicación elemento a elemento
            r = patch * f
            # Suma
            output[i, j] = torch.sum(r)
    return output

output = conv(x, f)
print(output)

tensor([[-45., 103.],
        [-96., 133.]])


### 2.1 Stride

Stride is the step of the kernel over the window.

When $S = 1$, the number of positions is:

$$
N - K + 1
$$

This comes from the fact that the last valid starting position is:

$$
N - K
$$

For example, in a $4 \times 4$ input with a $3 \times 3$ kernel:

$$
4 - 3 = 1
$$

Since indexing starts at $0$, we add $1$ to count all valid positions:

$$
(N - K) + 1
$$

---

When stride is greater than $1$, the kernel moves in steps of size $S$:

$$
0, S, 2S, 3S, \dots
$$

We want the largest index $i$ such that the kernel still fits:

$$
i \cdot S \leq N - K
$$

Solving for $i$:

$$
i \leq \frac{N - K}{S}
$$

The number of valid positions is (with floor):

$$
\left\lfloor \frac{N - K}{S} \right\rfloor + 1
$$

---

Therefore, the general formula (without padding) is:

$$
\text{output} = \left\lfloor \frac{N - K}{S} \right\rfloor + 1
$$

In [22]:
import math

def conv_output_size(h, w, k_h, k_w, stride=1):
    o_h = math.floor((h - k_h) / stride) + 1
    o_w = math.floor((w - k_w) / stride) + 1
    return o_h, o_w

## 2. Activation function

In [86]:
relu = torch.nn.ReLU()
z = relu(output)
z

tensor([[  0., 103.],
        [  0., 133.]])

# 3. Pooling

Reduce the size of the feature map. Similar to convolution, it slides a window across the imagen

In the next example lets assume a stride of 2.

<div>
    <img src='../images/POOLLAYER.png' width="600">
</div>

In [111]:
import torch

x = torch.tensor(
    [
        [45, 12, 5, 17],
        [22, 10, 35, 6],
        [88, 26, 51, 19],
        [9, 77, 42, 3]
    ], 
    dtype=torch.float32
)

# Pool size
ps = 2

h, w = x.shape

stride = 2
po_h, po_w = conv_output_size(h, w, ps, ps, stride=stride)

p_max = torch.zeros((po_h, po_w))
p_avg = torch.zeros((po_h, po_w))

for i in range(po_h):
    for j in range(po_w):
        ii = i * stride
        jj = j * stride
        e = x[ii:ii+ps, jj:jj+ps]
        p_max[i, j] = torch.max(e)
        p_avg[i, j] = torch.mean(e)

print("Max pooling")
print(p_max)
print("Average pooling")
print(p_avg)

Max pooling
tensor([[45., 35.],
        [88., 51.]])
Average pooling
tensor([[22.2500, 15.7500],
        [50.0000, 28.7500]])


## 4. Flatten + Fully Connected (Linear)

Finally the result is flattened and pass to a fully connected layer

<div>
    <img src='../images/FLATTEN.png' width="600">
</div>

In [112]:
p_max.flatten()

tensor([45., 35., 88., 51.])

## Flow

Input → Conv → ReLU → Pool → Flatten → FC → Output

# RGB Images

Now, we have only considered a single channel. However, RGB images have three channels.

The key difference is that instead of using a kernel with a single channel, we now use a kernel with three channels (or layers), one for each input channel. In general a kernel with n channels is called a `filter`.

<div>
    <img src='../images/CONVRGB.png' width="600">
</div>

In [ ]:
R = torch.Tensor([
    [45, 12, 5, 17],
    [22, 10, 35, 6],
    [88, 26, 51, 19],
    [9, 77, 42, 3]
])

G = torch.Tensor([
    [15, 13, 6, 7],
    [23, 14, 3, 6],
    [12, 16, 1, 34],
    [91, 66, 22, 31]
])

B = torch.Tensor([
    [6, 12, 5, 17],
    [52, 14, 38, 8],
    [88, 66, 5, 9],
    [70, 71, 72, 73]
])

KR = torch.Tensor([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

KG = torch.Tensor([
    [0, -1, 0],
    [-1, 4, -1],
    [0, -1, 0]
])

KB = torch.Tensor([
    [0, -1, 0],
    [-1, 3, -1],
    [0, -1, 0]
])


h, w = R.shape
k_h, k_w = KR.shape

stride = 1
o_h, o_w = conv_output_size(h, w, k_h, k_w, stride=1)

output = torch.zeros((o_h, o_w))

for i in range(o_h):
    for j in range(o_w):
        ii = i * stride
        jj = j * stride
        PR = R[ii:ii+k_h, jj:jj+k_w]
        PG = G[ii:ii+k_h, jj:jj+k_w]
        PB = B[ii:ii+k_h, jj:jj+k_w]
        rR = PR * KR
        rG = PG * KG
        rB = PB * KB
        val_R = torch.sum(rR)
        val_G = torch.sum(rG)
        val_B = torch.sum(rB)
        # Equivalent to (sum is distributed) 
        # output[i, j] = torch.sum(rR + rG + rB)        
        output[i, j] = val_R + val_G + val_B

print(output)

tensor([[-170.,  170.],
        [-105., -108.]])


In a compact way

In [40]:
import torch

# Shape (channels, height, width)
X = torch.Tensor([
    [
        [45, 12, 5, 17],
        [22, 10, 35, 6],
        [88, 26, 51, 19],
        [9, 77, 42, 3]
    ],
    [
        [15, 13, 6, 7],
        [23, 14, 3, 6],
        [12, 16, 1, 34],
        [91, 66, 22, 31]
    ],
    [
        [6, 12, 5, 17],
        [52, 14, 38, 8],
        [88, 66, 5, 9],
        [70, 71, 72, 73]
    ]
])

# Shape (channels, k, k)
F = torch.Tensor([
    [
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ],
    [
        [0, -1, 0],
        [-1, 4, -1],
        [0, -1, 0]
    ],
    [
        [0, -1, 0],
        [-1, 3, -1],
        [0, -1, 0]
    ]
])

h, w = X.shape[-2:]
k_h, k_w = F.shape[-2:]

stride = 1
o_h, o_w = conv_output_size(h, w, k_h, k_w, stride=1)

output = torch.zeros((o_h, o_w))

for i in range(o_h):
    for j in range(o_w):
        ii = i * stride
        jj = j * stride
        P = X[:, ii:ii+k_h, jj:jj+k_w]
        val = torch.sum(P * F)  
        output[i, j] = val

print(output)

tensor([[-170.,  170.],
        [-105., -108.]])


# Multi channel

Before:

`
RGB (3 channels) → 1 filter (3 layers) → 1 map
`

Now:

`
RGB (3 channels) → N filters (3 layers) → N map
`

<div>
    <img src='../images/MULTIFILTERS.png' width="900">
</div>

So we just add the multi filter dimension, lets say 2 filters of 3 layers and a K kernel: (2, 3, K, K)

In [54]:
import torch

# Shape (channels, height, width)
X = torch.Tensor([
    [
        [45, 12, 5, 17],
        [22, 10, 35, 6],
        [88, 26, 51, 19],
        [9, 77, 42, 3]
    ],
    [
        [15, 13, 6, 7],
        [23, 14, 3, 6],
        [12, 16, 1, 34],
        [91, 66, 22, 31]
    ],
    [
        [6, 12, 5, 17],
        [52, 14, 38, 8],
        [88, 66, 5, 9],
        [70, 71, 72, 73]
    ]
])

# Shape (filters, channels, k, k)
Fs = torch.Tensor([
    [
        [
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ],
        [
            [0, -1, 0],
            [-1, 4, -1],
            [0, -1, 0]
        ],
        [
            [0, -1, 0],
            [-1, 3, -1],
            [0, -1, 0]
        ]
    ],
    [
        [
            [0, -1, 0],
            [-1, 2, -1],
            [0, -1, 0]
        ],
        [
            [0, -1, 0],
            [-1, 1, -1],
            [0, -1, 0]
        ],
        [
            [0, -1, 0],
            [-1, 0, -1],
            [0, -1, 0]
        ]
    ]
])

h, w = X.shape[-2:]
k_h, k_w = Fs.shape[-2:]

stride = 1
o_h, o_w = conv_output_size(h, w, k_h, k_w, stride=1)

C_out = Fs.shape[0]
output = torch.zeros((C_out, o_h, o_w))

for f in range(C_out):
    F = Fs[f]
    for i in range(o_h):
        for j in range(o_w):
            ii = i * stride
            jj = j * stride
            P = X[:, ii:ii+k_h, jj:jj+k_w]
            val = torch.sum(P * F)  
            output[f, i, j] = val

print(output)

tensor([[[-170.,  170.],
         [-105., -108.]],

        [[-284.,  -58.],
         [-429., -279.]]])


# Padding

Padding consists of adding zeros around the input.

<div>
    <img src='../images/PADDING.png' width="600">
</div>

It prevents the spatial dimensions from shrinking too quickly and helps preserve information near the borders.

For example, without padding:

5×5 → 3×3 → 1×1 → eventually vanishes

---

## Formula

$$
o = \left\lfloor \frac{N + 2P - K}{S} \right\rfloor + 1
$$

For 2D inputs:

$$
o_h = \left\lfloor \frac{H + 2P - K_h}{S} \right\rfloor + 1
$$

$$
o_w = \left\lfloor \frac{W + 2P - K_w}{S} \right\rfloor + 1
$$

---

## Types of padding

- **Valid padding** ($P = 0$): No padding  
- **Same padding** (output size equals input size):

$$
P = \frac{K - 1}{2}
$$

This only works when:
- stride = 1  
- the kernel size is **odd** (e.g., 3, 5, 7, ...)


Note that padding does not add new information; it only controls how the kernel interacts with the borders.

**Note:** Padding will not be implemented in this work. The previous formulations assume $P = 0$, unless stated otherwise.
